# 05 - EDA and Feature Engineering**Objective:** Explore Wine data patterns and create new features.**Steps:**1. Statistical summary and class distribution2. Visualizations (distributions, correlations, feature comparison by cultivar)3. Feature scaling4. Feature engineering (interaction and ratio features)5. Save engineered data

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import Pathimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)print("Libraries imported successfully")

In [ ]:
# TODO: Load the clean data and inspect its structure# Start by reading data/processed/clean_data.csv into a DataFrame.# Then use .info() to see column dtypes and non-null counts,# .head() to preview a few rows, and .describe() for summary statistics.# The dataset has 150 rows, 4 numeric features, and a 3-class target.PROCESSED_DIR = Path("../data/processed")# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")# print(f"Shape: {df.shape}")# print(df.info())# print(df.head())# print(df.describe())

### Statistical Summary & DistributionBefore building any models, it is important to understand the range and spread of your data.The target variable — `class` — has three categories (a multi-class classification problem).The three cultivars have 59, 71, and 48 samples, so the dataset is slightly imbalanced.Wine has 13 continuous features measuring chemical properties of the wines.

In [ ]:
# TODO: Examine the target variable distribution and feature summary# Use df.describe() to see min, max, mean, and quartiles for all 13 features.# Check the class distribution with df["class"].value_counts() — each species has 50 rows.## print(df.describe())# print(df["class"].value_counts())

#### A little primer on groupby - `groupby` is a powerful pandas method that allows you to split your data into groups based on some criteria, apply a function to each group, and then combine the results. For example, to see how average alcohol differs between cultivars, you can do:```pythonalcohol_by_cultivar = df.groupby("class")["alcohol"].mean()```- `aggregate` is a method that allows you to apply multiple functions to your grouped data. For example, to get both the mean and standard deviation of color intensity by cultivar, you can do:```pythonstats_by_species = df.groupby("class")["color_intensity"].aggregate(["mean", "std"])```Aggregate functions can be any function that takes a Series and returns a single value, such as `mean`, `std`, `min`, `max`, etc.Aggregate can be deployed on multiple columns at once, and you can specify different functions for each column if needed.

In [ ]:
# === Executed Example: GroupBy and Aggregate ===# Small inline dataset showing how groupby splits data by class# and compares flower measurements between species.import pandas as pddata = pd.DataFrame({    "class": [0, 0, 0, 1, 1, 1],    "color_intensity": [5.1, 4.9, 5.0, 6.0, 6.2, 5.9],    "alcohol": [1.4, 1.5, 1.3, 4.5, 4.7, 4.4],})mean_by_class = data.groupby("class")["alcohol"].mean()print("Average alcohol by class:\n", mean_by_class)stats_by_class = data.groupby("class")["color_intensity"].agg(["mean", "std", "min", "max"])print("\nColor intensity statistics by class:\n", stats_by_class)

In [ ]:
# === Commented Template: GroupBy and Aggregate ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "group_col": [val1, val1, val2, val2],#     "value_col": [10, 20, 30, 40],# })# grouped = data.groupby("group_col")["value_col"].mean()# stats = data.groupby("group_col")["value_col"].agg(["mean", "std", "min", "max"])

### Missing Value Imputation

Real-world datasets often have missing values.
Our clean data does not, but this section shows the tools to handle them.

Common strategies:
- **Drop rows**: `df.dropna()` — fast, loses samples
- **Mean/Median imputation**: `SimpleImputer(strategy='median')` — preserves sample count
- **KNN imputation**: `KNNImputer()` — estimates from neighbors, more robust
- **Forward fill**: `df.ffill()` — for sequential data

In [ ]:
# === Executed Example: Missing Value Imputation ===
# Using SimpleImputer to fill missing values with the mean.

from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "color_intensity": [5.1, np.nan, 5.0, 6.0, np.nan],
    "alcohol": [1.4, 1.5, np.nan, 4.5, 4.7],
})

print("Before imputation:")
print(data)

imputer = SimpleImputer(strategy='mean')
imputed = imputer.fit_transform(data)
imputed_df = pd.DataFrame(imputed, columns=data.columns)
print("\nAfter imputation (mean strategy):")
print(imputed_df)
print(f"\nImputed color_intensity: {imputer.statistics_[0]:.3f}")
print(f"Imputed alcohol: {imputer.statistics_[1]:.3f}")

In [ ]:
# TODO: Compare feature distributions by cultivar# Which measurements best separate the three species?# Use boxplots to compare all 4 features.# Cultivar_1 should separate clearly on certain chemical features.## features_to_plot = ["color_intensity", "hue", "alcohol", "proline"]# fig, axes = plt.subplots(2, 2, figsize=(12, 10))# for ax, feature in zip(axes.ravel(), features_to_plot):#     sns.boxplot(x="class", y=feature, data=df, ax=ax)#     ax.set_title(f"{feature} by class")# plt.tight_layout()# plt.show()

### VisualizationsVisual exploration helps you spot patterns and relationships that summary statistics miss.Focus on:- How each feature is distributed (histograms)- Which features correlate with each other and with the target (heatmap)- Which features separate the three species (boxplots)

In [ ]:
# TODO: Plot histograms for all numerical features# Wine has 13 features; select the most informative ones to plot.# Use df[features].hist() with bins=15 and a large figure size.# features_to_plot = ["color_intensity", "hue", "alcohol", "proline"]# df[features_to_plot].hist(bins=15, figsize=(12, 8))# plt.tight_layout()# plt.show()

In [ ]:
# TODO: Create a correlation heatmap# Use df.corr() to compute pairwise correlations between numeric columns.# With 13 features + the target, the full heatmap is easy to read.# Then visualize with sns.heatmap().## TODO: Identify the features most correlated with species# Sort the correlation values for the class column to see which features# best distinguish the three species.# plt.figure(figsize=(8, 6))# sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")# plt.title("Feature Correlation Heatmap")# plt.show()## target_corr = df.corr()["class"].sort_values(ascending=False)# print("Correlations with class:")# print(target_corr)

### Feature ScalingMany machine learning algorithms (SVM, logistic regression, neural networks) are sensitive tothe scale of input features. StandardScaler transforms each feature to have mean 0 andstandard deviation 1, which puts all features on equal footing.Tree-based models (Random Forest, XGBoost) do not require scaling since they split onthresholds independently of feature magnitude.

In [ ]:
# === Executed Example: Feature Scaling ===# Small inline dataset showing how StandardScaler transforms features# to have mean ~0 and std ~1.from sklearn.preprocessing import StandardScalerimport pandas as pddata = pd.DataFrame({    "color_intensity": [5.1, 4.9, 5.0, 6.0, 6.2],    "alcohol": [1.4, 1.5, 1.3, 4.5, 4.7],    "class": [0, 0, 0, 1, 1],})scaler = StandardScaler()scaled_features = scaler.fit_transform(data[["color_intensity", "alcohol"]])scaled_df = pd.DataFrame(scaled_features, columns=["color_intensity_scaled", "alcohol_scaled"])scaled_df["class"] = data["class"]print(scaled_df)print(f"Means after scaling: {scaled_df[['color_intensity_scaled', 'alcohol_scaled']].mean().values}")print(f"Stds after scaling: {scaled_df[['color_intensity_scaled', 'alcohol_scaled']].std().values}")

In [ ]:
from sklearn.preprocessing import StandardScaler# TODO: Scale the 4 numeric features# Wine has only continuous features with different units and scales.# StandardScaler normalises them so SVM / logistic regression aren't biased# by the absolute magnitude of each measurement.## Create a StandardScaler, call fit_transform() to compute the mean and std# and return the scaled array in one step.## After scaling, verify that each feature has mean ~0 and std ~1.# numeric_features = ["color_intensity", "hue", "alcohol", "proline"]# scaler = StandardScaler()# df_scaled = scaler.fit_transform(df[numeric_features])## Rename the scaled columns with a "_scaled" suffix# df_scaled = pd.DataFrame(df_scaled, columns=[f"{col}_scaled" for col in numeric_features])# print(f"Means after scaling: {df_scaled.mean().values}")# print(f"Stds after scaling: {df_scaled.std().values}")# print(f"All means near zero: {np.allclose(df_scaled.mean(), 0, atol=1e-10)}")

### Feature EngineeringNew features derived from existing columns can capture interactions and non-linear relationships.Good candidates for Wine:- **Phenol ratio**: `alcohol × proline` — proportion of flavanoid phenols- **Color-hue product**: `color_intensity × hue` — combined color metric- **Alcohol-color ratio**: `alcohol / color_intensity` — relative flower proportionBe careful with division by zero — add a small epsilon or +1 to the denominator.#### NoteIn pandas, you can create interaction features like this:```pythondf["feature1_feature2"] = df["feature1"] * df["feature2"]```

In [ ]:
# === Executed Example: Interaction Features ===# Multiplication and ratio on a small inline dataset.import pandas as pddata = pd.DataFrame({    "color_intensity": [5.1, 4.9, 6.0, 6.2, 5.9],    "hue": [3.5, 3.0, 3.3, 2.9, 3.0],    "alcohol": [1.4, 1.5, 4.5, 4.7, 4.4],    "proline": [0.2, 0.2, 1.5, 1.6, 1.4],})data["phenol_ratio"] = data["alcohol"] * data["proline"]print("Phenol ratio:\n", data[["alcohol", "proline", "phenol_ratio"]])data["alcohol_color_ratio"] = data["alcohol"] / data["color_intensity"]print("\nAlcohol / Color ratio:\n", data[["alcohol", "color_intensity", "alcohol_color_ratio"]])

In [ ]:
# === Commented Template: Interaction Features ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "feature_a": [val1, val2, val3],#     "feature_b": [val1, val2, val3],# })# data["a_times_b"] = data["feature_a"] * data["feature_b"]# EPS = 1e-6# data["a_over_b"] = data["feature_a"] / (data["feature_b"] + EPS)

In [ ]:
# TODO: Create new features based on flower morphology# The Wine dataset has chemical measurements. Ratio and product# features can make species separation more pronounced.## Examples:#   df["phenol_ratio"] = df["alcohol"] * df["proline"]#   df["color_hue_product"] = df["color_intensity"] * df["hue"]#   df["alcohol_color_ratio"] = df["alcohol"] / df["color_intensity"]## TODO: Verify the new features# Check that they have finite values and reasonable ranges with .describe().

In [ ]:
# TODO: Save the engineered data for the next notebook# Include both the original and new features.PROCESSED_DIR = Path("../data/processed")PROCESSED_DIR.mkdir(parents=True, exist_ok=True)# df.to_csv(PROCESSED_DIR / "engineered_data.csv", index=False)# print("Engineered data saved to data/processed/engineered_data.csv")

### Unsupervised Clustering (KMeans)

Clustering groups observations without using the target labels.
We use **KMeans** which partitions data into $k$ clusters by minimizing
within-cluster variance.

**Questions:**
- Do the clusters found by KMeans align with the actual class classes?
- How many natural groups exist in the data?


In [ ]:
# TODO: Apply KMeans clustering and compare with target labels
# Clustering groups data without using labels.
# Use the elbow method to find optimal k, then compare clusters vs target.

# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import adjusted_rand_score
# from sklearn.decomposition import PCA

# Step 1: Scale features
# X_clust = df.drop("class", axis=1).select_dtypes(include=[np.number])
# X_clust_scaled = StandardScaler().fit_transform(X_clust)
#
# Step 2: Elbow method for k=2..10
# inertias = []
# for k in range(2, min(11, X_clust_scaled.shape[1] + 1)):
#     kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
#     kmeans.fit(X_clust_scaled)
#     inertias.append(kmeans.inertia_)
#
# Step 3: Plot elbow curve
# plt.plot(range(2, min(11, X_clust_scaled.shape[1] + 1)), inertias, 'bo-')
# plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow Method')
# plt.show()
#
# Step 4: Fit KMeans and compare with target
# df["Cluster"] = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_clust_scaled)
# ari = adjusted_rand_score(df["class"], df["Cluster"])
# print(f"Adjusted Rand Index: {ari:.4f}")
# print(pd.crosstab(df["Cluster"], df["class"]))
#
# Step 5: Visualize via PCA
# pca_vis = PCA(n_components=2, random_state=42)
# X_pca_vis = pca_vis.fit_transform(X_clust_scaled)
# plt.scatter(X_pca_vis[:, 0], X_pca_vis[:, 1], c=df['Cluster'], cmap='viridis', edgecolors='k', alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('KMeans Clusters')
# plt.show()


### Principal Component Analysis (PCA)

PCA finds orthogonal axes (principal components) that capture the maximum variance
in the data. It is useful for:
- **Dimensionality reduction**: compressing many features into fewer components
- **Visualization**: projecting high-dimensional data to 2D or 3D
- **Noise reduction**: discarding low-variance components

PCA is **unsupervised** — it does not use the target labels.


In [ ]:
# TODO: Apply PCA for dimensionality reduction and visualization
# PCA finds the axes of maximum variance in the data.
# It can reduce high-dimensional data to 2D for visualization.

# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# X = df.drop("class", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
#
# Step 1: Fit PCA with min(n_features, 10) components
# n_comps = min(X.shape[1], 10)
# pca = PCA(n_components=n_comps, random_state=42)
# X_pca = pca.fit_transform(X_scaled)
#
# Step 2: Scree plot
# plt.bar(range(1, n_comps + 1), pca.explained_variance_ratio_)
# plt.xlabel('PC'); plt.ylabel('Explained Variance Ratio')
# plt.title('Scree Plot'); plt.show()
#
# Step 3: Cumulative explained variance
# cumulative = np.cumsum(pca.explained_variance_ratio_)
# plt.plot(range(1, n_comps + 1), cumulative, 'bo-')
# plt.axhline(y=0.95, color='r', linestyle='--', label='95%')
# plt.legend(); plt.show()
#
# Step 4: 2D PCA scatter colored by target
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["class"], cmap="coolwarm", edgecolors="k", alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2')
# plt.title('PCA Projection'); plt.colorbar()
# plt.show()
#
# Step 5: Inspect feature loadings
# loadings = pd.DataFrame(
#     pca.components_.T[:, :3],
#     index=X.columns,
#     columns=['PC1', 'PC2', 'PC3'],
# )
# print(loadings['PC1'].abs().sort_values(ascending=False).head(5))


### Linear Discriminant Analysis (LDA)

LDA finds axes that **maximize class separability**. Unlike PCA (unsupervised),
LDA uses the target labels to find projections that best separate the classes.

**LDA vs PCA**:
- PCA maximizes **variance** (ignores labels)
- LDA maximizes **class separation** (uses labels)
- For classification, LDA often gives better separation in fewer components


In [ ]:
# TODO: Apply LDA for supervised dimensionality reduction
# LDA uses the target labels to maximize class separation.
# Compare its projection with PCA (unsupervised).

# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# X = df.drop("class", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
# y = df["class"]
#
# Step 1: Fit LDA
# n_classes = y.nunique()
# lda = LDA(n_components=min(n_classes - 1, X_scaled.shape[1]))
# X_lda = lda.fit_transform(X_scaled, y)
# print(f"LDA reduced to {X_lda.shape[1]} component(s)")
#
# Step 2: Visualize
# if X_lda.shape[1] >= 2:
#     plt.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
#     plt.xlabel('LD1'); plt.ylabel('LD2')
# else:
#     for cls in sorted(y.unique()):
#         plt.hist(X_lda[y == cls, 0], bins=20, alpha=0.5, label=f'Class {cls}')
#     plt.legend(); plt.xlabel('LD1'); plt.ylabel('Frequency')
# plt.title('LDA Projection'); plt.show()
#
# Step 3: Side-by-side PCA vs LDA
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
# ax1.set_title('PCA (unsupervised)'); ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')
# x_lda_2d = X_lda[:, :min(2, X_lda.shape[1])]
# if x_lda_2d.shape[1] == 1:
#     x_lda_2d = np.hstack([x_lda_2d, np.zeros_like(x_lda_2d)])
# ax2.scatter(x_lda_2d[:, 0], x_lda_2d[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
# ax2.set_title('LDA (supervised)'); ax2.set_xlabel('LD1'); ax2.set_ylabel('LD2')
# plt.tight_layout(); plt.show()


### Recursive Feature Elimination (RFE)

RFE recursively removes the least important features, building a model at each step.
It ranks features by importance and finds the optimal subset.

**Benefits:**
- Reduces overfitting by removing noisy features
- Improves model interpretability
- Can speed up training and prediction


In [ ]:
# TODO: Apply RFE for feature selection
# RFE recursively removes the least important features.
# RFECV uses cross-validation to find the optimal subset.

# from sklearn.feature_selection import RFE, RFECV
# from sklearn.linear_model import LogisticRegression

# X = df.drop("class", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
# y = df["class"]
#
# Step 1: Fit RFE
# estimator = LogisticRegression(max_iter=1000, random_state=42)
# rfe = RFE(estimator=estimator, n_features_to_select=min(10, X_scaled.shape[1]), step=1)
# rfe.fit(X_scaled, y)
#
# Step 2: Display feature rankings
# ranking_df = pd.DataFrame({
#     'Feature': X.columns,
#     'Rank': rfe.ranking_,
#     'Selected': rfe.support_,
# }).sort_values('Rank')
# print(ranking_df)
#
# Step 3: RFECV for automatic feature count
# rfecv = RFECV(estimator=estimator, step=1, cv=5, scoring='accuracy')
# rfecv.fit(X_scaled, y)
# print(f"Optimal features: {rfecv.n_features_}")
#
# Step 4: Plot CV accuracy vs feature count
# plt.plot(range(len(rfecv.cv_results_['mean_test_score'])),
#          rfecv.cv_results_['mean_test_score'], 'bo-')
# plt.axvline(x=rfecv.n_features_, color='r', linestyle='--')
# plt.title('RFE: Optimal Feature Count'); plt.show()


### Exercises1. **Try different scalers**: Replace `StandardScaler` with `MinMaxScaler` or `RobustScaler` and compare how the scaled distributions look.2. **Ratio by class**: Use `df.groupby("class")` on `alcohol_color_ratio` to see if the ratio alone can separate all three species. Which species has the largest ratio?3. **Log transform features**: Some features may benefit from log transform for Wine, but try `np.log1p()` on `phenol_ratio` anyway. How does the distribution change?4. **Pairplot**: Use `sns.pairplot()` on all 4 original features, coloring by class — do the three species form distinct clusters? Which two features separate them best?5. **Redundant features**: Check if `alcohol` and `proline` are highly correlated. If you had to drop one, which would you keep and why?